**LOAD DATA (STAGING)**

In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("../data/telecom_churn.csv")
if not data_path.exists():
    data_path = Path("data/telecom_churn.csv")

df = pd.read_csv(data_path)
df.columns = df.columns.str.lower()

print(df.shape)

(243553, 14)


In [2]:
df.head()

,customer_id,telecom_partner,gender,age,state,city,pincode,date_of_registration,num_dependents,estimated_salary,calls_made,sms_sent,data_used,churn
0,1,Reliance Jio,F,25,Karnataka,Kolkata,755597,2020-01-01,4,124962,44,45,-361,0
1,2,Reliance Jio,F,55,Mizoram,Mumbai,125926,2020-01-01,2,130556,62,39,5973,0
2,3,Vodafone,F,57,Arunachal Pradesh,Delhi,423976,2020-01-01,0,148828,49,24,193,1
3,4,BSNL,M,46,Tamil Nadu,Kolkata,522841,2020-01-01,1,38722,80,25,9377,1
4,5,BSNL,F,26,Tripura,Delhi,740247,2020-01-01,2,55098,78,15,1393,0


**DATA PREPROCESSING (KDD – Preprocessing)**

Handle Dates

In [3]:
df['date_of_registration'] = pd.to_datetime(df['date_of_registration'])
df['registration_year'] = df['date_of_registration'].dt.year
df['registration_month'] = df['date_of_registration'].dt.month


Missing Values

In [4]:
df.fillna({
    'calls_made': 0,
    'sms_sent': 0,
    'data_used': 0,
    'estimated_salary': df['estimated_salary'].median()
}, inplace=True)


Encode Categorical Columns

In [5]:
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

**ETL → POSTGRESQL DATA WAREHOUSE**

In [ ]:
import os
from sqlalchemy import URL, create_engine, text

# Configure these values through environment variables instead of storing credentials here.
db_user = os.environ["TELECOM_DB_USER"]
db_password = os.environ["TELECOM_DB_PASSWORD"]
db_host = os.getenv("TELECOM_DB_HOST", "localhost")
db_port = int(os.getenv("TELECOM_DB_PORT", "5432"))
db_name = "telecom_dw"

admin_url = URL.create(
    "postgresql+psycopg2",
    username=db_user,
    password=db_password,
    host=db_host,
    port=db_port,
    database="postgres",
)
admin_engine = create_engine(admin_url, isolation_level="AUTOCOMMIT")

with admin_engine.connect() as connection:
    database_exists = connection.execute(
        text("SELECT 1 FROM pg_catalog.pg_database WHERE datname = :database_name"),
        {"database_name": db_name},
    ).scalar()
    if not database_exists:
        connection.execute(text(f'CREATE DATABASE "{db_name}"'))
        print(f"Database '{db_name}' created.")
    else:
        print(f"Database '{db_name}' already exists.")

engine_url = URL.create(
    "postgresql+psycopg2",
    username=db_user,
    password=db_password,
    host=db_host,
    port=db_port,
    database=db_name,
)
engine = create_engine(engine_url)
df.to_sql("stg_telecom_churn", engine, if_exists="replace", index=False)
print("Data loaded into staging table 'stg_telecom_churn' successfully.")

**DATA MINING – CLASSIFICATION MODEL**

Feature Selection

In [6]:
X = df[['age', 'num_dependents', 'estimated_salary',
        'calls_made', 'sms_sent', 'data_used']]
y = df['churn']


Train-Test Split

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)


Logistic regression

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


Accuracy: 0.5112911614157063
Precision: 0.20213909795710486
Recall: 0.4875119421318411
ROC-AUC: 0.5030772617015181


logistic regression witrh feature scaling


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


Accuracy: 0.5004379602003668
Precision: 0.2018451292409313
Recall: 0.5046403712296984
ROC-AUC: 0.5034173941684891


Decision tree

In [11]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    max_depth=6,
    class_weight='balanced',
    random_state=42
)

dt.fit(X_train, y_train)

y_pred = dt.predict(X_test)
y_prob = dt.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


Accuracy: 0.685380341061506
Precision: 0.20047441058079357
Recall: 0.19032346117101132
ROC-AUC: 0.500142479968514


**CLUSTERING MODEL (KMEANS)**

In [12]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

print("Silhouette Score:", silhouette_score(X_scaled, clusters))


Silhouette Score: 0.1087064481133702
